# Full Embedding Integration in Colab

**Workflow:** All processing in Colab, upload complete artifacts to Droplet

## Steps:
1. Download current embeddings/index from Droplet
2. Fetch pending plays from API
3. Generate embeddings in Colab (GPU accelerated)
4. Merge and rebuild index in Colab (abundant RAM)
5. Upload complete files back to Droplet
6. Restart API service

**Requirements:**
- Enable GPU runtime (Runtime → Change runtime type → T4 GPU)
- SSH key for Droplet access (for rsync)

In [ ]:
# Install dependencies
!pip install -q sentence-transformers requests joblib paramiko

In [ ]:
import torch

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️  WARNING: GPU not available. Enable GPU: Runtime → Change runtime type → GPU")

In [ ]:
# Configuration
API_URL = "http://cratemusic.duckdns.org"
DROPLET_HOST = "root@cratemusic.duckdns.org"
REMOTE_DATA_DIR = "/root/faiss-search-api/data"
MODEL_NAME = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
BATCH_SIZE = 1024 if device == 'cuda' else 32

print(f"API URL: {API_URL}")
print(f"Model: {MODEL_NAME}")
print(f"Batch size: {BATCH_SIZE}")

## Step 1: Download Current Data from Droplet

In [ ]:
# Download current embeddings and index via HTTP
import requests
from pathlib import Path

# Create local data directory
!mkdir -p data

print("Downloading current embeddings and metadata...")

# For now, use API to get metadata about current state
response = requests.get(f"{API_URL}/api/health")
health = response.json()

print(f"Current state:")
print(f"  Total vectors: {health['total_vectors']:,}")
print(f"  Embedding dimension: {health['embedding_dimension']}")
print(f"  Memory usage: {health['memory_usage_mb']:.1f} MB")

In [ ]:
# Setup SSH for file transfer
# Option 1: Manual SCP (you run this in your local terminal, then upload to Colab)
# scp root@cratemusic.duckdns.org:/root/faiss-search-api/data/embeddings_256d.npy ./data/
# scp root@cratemusic.duckdns.org:/root/faiss-search-api/data/play_ids.npy ./data/
# scp root@cratemusic.duckdns.org:/root/faiss-search-api/data/pca_transformer_256d.joblib ./data/

# Option 2: Upload files to Google Drive, mount in Colab
# from google.colab import drive
# drive.mount('/content/drive')

# For this notebook, we'll use wget to download via a temporary HTTP endpoint
# OR we can use the API to get data incrementally

print("📝 Manual step required:")
print("  1. Run locally: scp root@cratemusic.duckdns.org:/root/faiss-search-api/data/embeddings_256d.npy ./")
print("  2. Run locally: scp root@cratemusic.duckdns.org:/root/faiss-search-api/data/play_ids.npy ./")
print("  3. Run locally: scp root@cratemusic.duckdns.org:/root/faiss-search-api/data/pca_transformer_256d.joblib ./")
print("  4. Upload files to this Colab session (Files panel on left)")
print("\nAlternatively, upload to Google Drive and mount it.")

In [ ]:
# Verify files are present
import numpy as np
import joblib
from pathlib import Path

# Check if files exist
embeddings_path = Path('embeddings_256d.npy')
play_ids_path = Path('play_ids.npy')
pca_path = Path('pca_transformer_256d.joblib')

if not all([embeddings_path.exists(), play_ids_path.exists(), pca_path.exists()]):
    print("❌ Files not found. Please upload them to Colab.")
    print(f"  embeddings_256d.npy: {'✓' if embeddings_path.exists() else '✗'}")
    print(f"  play_ids.npy: {'✓' if play_ids_path.exists() else '✗'}")
    print(f"  pca_transformer_256d.joblib: {'✓' if pca_path.exists() else '✗'}")
else:
    # Load with mmap to check size without loading into memory
    existing_embeddings = np.load(embeddings_path, mmap_mode='r')
    existing_ids = np.load(play_ids_path, mmap_mode='r')
    pca = joblib.load(pca_path)
    
    print(f"✓ Files loaded successfully")
    print(f"  Embeddings: {existing_embeddings.shape}")
    print(f"  Play IDs: {existing_ids.shape}")
    print(f"  PCA components: {pca.n_components}")

## Step 2: Fetch Pending Plays

In [ ]:
import requests
import json

print(f"Fetching pending plays from {API_URL}/api/embeddings/pending...")

response = requests.get(f"{API_URL}/api/embeddings/pending?limit=5000")
response.raise_for_status()

batch = response.json()

print(f"{'='*80}")
print(f"PENDING PLAYS")
print(f"{'='*80}")
print(f"Batch ID: {batch['batch_id']}")
print(f"Plays to embed: {len(batch['plays'])}")
print(f"Total pending: {batch['total_pending']}")
print(f"\nSample play:")
print(json.dumps(batch['plays'][0], indent=2))

## Step 3: Generate Embeddings (GPU Accelerated)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

# Load model
print(f"Loading model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"✓ Model loaded (embedding dim: {model.get_sentence_embedding_dimension()})\n")

# Extract texts and IDs
texts = [play['enriched_text'] for play in batch['plays']]
new_play_ids = [play['id'] for play in batch['plays']]

print(f"{'='*80}")
print(f"GENERATING EMBEDDINGS")
print(f"{'='*80}")
print(f"Texts: {len(texts)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {device}\n")

start_time = time.time()

# Step 1: Generate 768d embeddings (normalized)
print("Step 1: Generating 768d embeddings...")
embeddings_768d = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
    device=device
)

elapsed = time.time() - start_time
print(f"✓ Generated 768d embeddings: {embeddings_768d.shape}")
print(f"  Time: {elapsed:.1f}s ({len(texts) / elapsed:.1f} texts/sec)\n")

# Step 2: Apply PCA (768d → 256d)
print("Step 2: Applying PCA transformation (768d → 256d)...")
new_embeddings_256d = pca.transform(embeddings_768d)
print(f"✓ Reduced to 256d: {new_embeddings_256d.shape}")

# Step 3: Re-normalize
print("Step 3: Re-normalizing 256d embeddings...")
for i in range(len(new_embeddings_256d)):
    norm = np.linalg.norm(new_embeddings_256d[i])
    if norm > 0:
        new_embeddings_256d[i] = new_embeddings_256d[i] / norm

new_embeddings_256d = new_embeddings_256d.astype('float32')
print(f"✓ Normalized 256d embeddings")

total_time = time.time() - start_time
print(f"\n{'='*80}")
print(f"EMBEDDING GENERATION COMPLETE")
print(f"{'='*80}")
print(f"Total time: {total_time:.1f}s")
print(f"Final shape: {new_embeddings_256d.shape}")

## Step 4: Merge Arrays (Plenty of RAM in Colab)

In [ ]:
print("Merging embeddings and IDs...")

# Convert to proper arrays
new_ids_array = np.array(new_play_ids, dtype=np.int64)

# Check for duplicates
existing_ids_set = set(existing_ids.tolist())
duplicates = [id for id in new_play_ids if id in existing_ids_set]

if duplicates:
    print(f"⚠️  Found {len(duplicates)} duplicate IDs, filtering them out...")
    mask = [id not in existing_ids_set for id in new_play_ids]
    new_ids_array = new_ids_array[mask]
    new_embeddings_256d = new_embeddings_256d[mask]

print(f"Before merge: {len(existing_ids):,} existing")
print(f"Adding: {len(new_ids_array):,} new")

# Merge (Colab has plenty of RAM, so just concatenate)
merged_embeddings = np.vstack([existing_embeddings, new_embeddings_256d])
merged_ids = np.concatenate([existing_ids, new_ids_array])

print(f"After merge: {len(merged_ids):,} total")
print(f"Merged embeddings shape: {merged_embeddings.shape}")

# Free old arrays
del existing_embeddings, existing_ids, new_embeddings_256d, embeddings_768d
import gc
gc.collect()
print("✓ Freed old arrays")

## Step 5: Rebuild FAISS Index (No Memory Constraints)

In [ ]:
!pip install -q faiss-cpu  # or faiss-gpu if you want GPU acceleration

In [ ]:
import faiss
import time

print(f"{'='*80}")
print(f"BUILDING FAISS INDEX")
print(f"{'='*80}")

n_vectors, d = merged_embeddings.shape
print(f"Index dimensions: {n_vectors:,} vectors × {d}d")

# FAISS index configuration (same as production)
nlist = 1024  # Number of clusters

start_time = time.time()

# Create index
quantizer = faiss.IndexFlatIP(d)
index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

# Train on sample
sample_size = min(100000, n_vectors)
print(f"Training on {sample_size:,} sample vectors...")
sample = np.array(merged_embeddings[:sample_size], dtype=np.float32)
faiss.normalize_L2(sample)
index.train(sample)
print("✓ Index trained")

# Add all vectors (in batches for progress tracking)
batch_size = 100000  # Colab can handle larger batches
print(f"Adding vectors in batches of {batch_size:,}...")

for i in range(0, n_vectors, batch_size):
    batch_end = min(i + batch_size, n_vectors)
    print(f"  Batch {i//batch_size + 1}: adding vectors {i:,} to {batch_end:,}")
    
    batch = np.array(merged_embeddings[i:batch_end], dtype=np.float32)
    faiss.normalize_L2(batch)
    index.add(batch)

elapsed = time.time() - start_time
print(f"\n✓ Index built: {index.ntotal:,} vectors")
print(f"  Time: {elapsed:.1f}s")

# Set nprobe for search
index.nprobe = 10

print(f"\n{'='*80}")
print(f"INDEX BUILD COMPLETE")
print(f"{'='*80}")

## Step 6: Save Files Locally

In [ ]:
print("Saving files...")

# Save embeddings
np.save('embeddings_256d_new.npy', merged_embeddings)
print(f"✓ Saved embeddings: {merged_embeddings.shape}")

# Save play IDs
np.save('play_ids_new.npy', merged_ids)
print(f"✓ Saved play IDs: {merged_ids.shape}")

# Save FAISS index
faiss.write_index(index, 'embeddings_256d_new.index')
print(f"✓ Saved FAISS index: {index.ntotal:,} vectors")

# Show file sizes
import os
print(f"\nFile sizes:")
print(f"  embeddings_256d_new.npy: {os.path.getsize('embeddings_256d_new.npy') / 1e6:.1f} MB")
print(f"  play_ids_new.npy: {os.path.getsize('play_ids_new.npy') / 1e6:.1f} MB")
print(f"  embeddings_256d_new.index: {os.path.getsize('embeddings_256d_new.index') / 1e6:.1f} MB")

## Step 7: Upload to Droplet

**Manual Steps:**

1. Download files from Colab (Files panel → right-click → Download):
   - `embeddings_256d_new.npy`
   - `play_ids_new.npy`
   - `embeddings_256d_new.index`

2. Upload to Droplet via SCP:
   ```bash
   scp embeddings_256d_new.npy root@cratemusic.duckdns.org:/root/faiss-search-api/data/embeddings_256d.npy
   scp play_ids_new.npy root@cratemusic.duckdns.org:/root/faiss-search-api/data/play_ids.npy
   scp embeddings_256d_new.index root@cratemusic.duckdns.org:/root/faiss-search-api/data/embeddings_256d.index
   ```

3. Restart API service:
   ```bash
   ssh root@cratemusic.duckdns.org "cd /root/faiss-search-api && docker-compose restart api"
   ```

4. Verify:
   ```bash
   curl http://cratemusic.duckdns.org/api/health | jq
   ```

In [ ]:
print("📥 Download the following files from Colab:")
print("   1. embeddings_256d_new.npy")
print("   2. play_ids_new.npy")
print("   3. embeddings_256d_new.index")
print("\n📤 Then upload to Droplet with:")
print("   scp embeddings_256d_new.npy root@cratemusic.duckdns.org:/root/faiss-search-api/data/embeddings_256d.npy")
print("   scp play_ids_new.npy root@cratemusic.duckdns.org:/root/faiss-search-api/data/play_ids.npy")
print("   scp embeddings_256d_new.index root@cratemusic.duckdns.org:/root/faiss-search-api/data/embeddings_256d.index")
print("\n🔄 Restart API:")
print("   ssh root@cratemusic.duckdns.org 'cd /root/faiss-search-api && docker-compose restart api'")
print("\n✅ Verify:")
print("   curl http://cratemusic.duckdns.org/api/health | jq")

## Summary

This notebook processes embeddings entirely in Colab where RAM is abundant, then uploads the final artifacts to the Droplet.

**Advantages:**
- No memory constraints (Colab has 12-15GB RAM)
- GPU acceleration for embedding generation
- No Droplet downtime during processing
- Simple upload/restart workflow

**For Future Updates:**
Just run this notebook again with the latest data!